In [1]:
import pandas as pd
import xml.etree.ElementTree as ET
from xml.dom import minidom

# Load the TSV file
df = pd.read_csv('tlg0086.tlg034.tkatsch-ara.tsv', sep='\t')

# Filter out rows where 'Unit' might be a header or invalid
df = df[df['Unit'].str.contains(r'\d+\.\d+', na=False)]

# Create the root TEI element
tei = ET.Element("TEI", xmlns="http://www.tei-c.org/ns/1.0")

# --- Basic Header (Template based on your example) ---
tei_header = ET.SubElement(tei, "teiHeader")
file_desc = ET.SubElement(tei_header, "fileDesc")
title_stmt = ET.SubElement(file_desc, "titleStmt")
ET.SubElement(title_stmt, "title", {"xml:lang": "ara"}).text = "Die arabische übersetzung der Poetik des Aristoteles"
ET.SubElement(title_stmt, "author").text = "Aristotle"
ET.SubElement(title_stmt, "editor").text = "Jaroslaus Tkatsch"

# Text Body
text = ET.SubElement(tei, "text")
body = ET.SubElement(text, "body")
edition_div = ET.SubElement(body, "div", {"type": "edition", "xml:lang": "ara", "n": "urn:cts:arabicLit:tlg0086.tlg034.tkatsch-ara"})

# Helper variables to track hierarchy
current_chapter = None
last_chapter_n = None

for _, row in df.iterrows():
    unit = str(row['Unit']).strip()
    if '.' not in unit: continue
    
    chapter_n, subchapter_n = unit.split('.')
    
    # Create new chapter div if needed
    if chapter_n != last_chapter_n:
        current_chapter = ET.SubElement(edition_div, "div", {
            "type": "textpart", 
            "subtype": "chapter", 
            "n": chapter_n
        })
        last_chapter_n = chapter_n
        
    # Create subchapter div
    subchapter_div = ET.SubElement(current_chapter, "div", {
        "type": "textpart", 
        "subtype": "subchapter", 
        "xml:base": f"urn:cts:arabicLit:tlg0086.tlg034.tkatsch-ara:{chapter_n}",
        "n": subchapter_n
    })
    
    # Add paragraph with the Arabic text
    p = ET.SubElement(subchapter_div, "p")
    p.text = str(row['Arabic (full — Tkatsch)'])

# Format and save
xml_str = minidom.parseString(ET.tostring(tei)).toprettyxml(indent="  ")
with open("tlg0086.tlg034.tkatsch-ara.xml", "w", encoding="utf-8") as f:
    f.write(xml_str)

print("TEI XML successfully built as 'tlg0086.tlg034.tkatsch-ara.xml'")

TEI XML successfully built as 'tlg0086.tlg034.tkatsch-ara.xml'
